In [ ]:
import os
import sys

from pyspark.sql import SparkSession

# Use the Python executable from the active environment
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Prefer local loopback for Spark networking
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (
    SparkSession.builder
    .appName("PySparkDemo")
    .master("local[2]")
    .config("spark.pyspark.python", python_path)
    .config("spark.pyspark.driver.python", python_path)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-avro_2.13:4.0.1"
    )
    .getOrCreate()
)
print("SparkSession created successfully.")

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

schema = StructType([
    StructField("OrderID", IntegerType(), False),
    StructField("CustomerName", StringType(), True),
    StructField("Product", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Price", DoubleType(), True),
    StructField("City", StringType(), True)
])

In [ ]:
data = [
    (101, "Amit", "Laptop", 1, 55000.00, "Mumbai"),
    (102, "Neha", "Mouse", 2, 750.00, "Pune"),
    (103, "Rahul", "Keyboard", 1, 1500.00, "Delhi"),
    (104, "Priya", "Monitor", 2, 12000.00, "Mumbai"),
    (105, "Ravi", "Laptop", 1, 60000.00, "Bangalore"),
    (106, "Rajveer Pratap Singh Chauhan", "Laptop", 1, 60000.00, "Chikballapore, Bangalore")
]

df = spark.createDataFrame(data, schema)

df.show(truncate=False)
df.printSchema()

In [ ]:
from pyspark.sql.functions import col

sales_df = df.withColumn(
    "TotalAmount",
    col("Quantity") * col("Price")
)

sales_df.show(truncate=False)

In [ ]:
sales_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/csv/sales")

In [ ]:
csv_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("output/csv/sales")

csv_df.show(truncate=False)
csv_df.printSchema()

In [ ]:
csv_df = spark.read \
    .option("header", True) \
    .schema(sales_df.schema) \
    .csv("output/csv/sales")

csv_df.show(truncate=False)

In [ ]:
sales_df.write \
    .mode("overwrite") \
    .json("output/json/sales")

In [ ]:
json_df = spark.read \
    .json("output/json/sales")

json_df.show()
json_df.printSchema()

In [ ]:
json_df = spark.read \
    .schema(sales_df.schema) \
    .json("output/json/sales")

json_df.show()

In [ ]:
nested_data = [
    {
        "OrderID": 201,
        "Customer": {
            "Name": "Amit",
            "City": "Mumbai"
        },
        "Items": [
            {"Product": "Laptop", "Quantity": 1},
            {"Product": "Mouse", "Quantity": 2}
        ]
    }
]

nested_df = spark.createDataFrame(nested_data)

nested_df.printSchema()
nested_df.show(truncate=False)

In [ ]:
nested_df.select(
    col("OrderID"),
    col("Customer.Name").alias("CustomerName"),
    col("Customer.City").alias("City")
).show()

In [ ]:
from pyspark.sql.functions import explode

items_df = nested_df.select(
    col("OrderID"),
    explode(col("Items")).alias("Item")
)

items_df.select(
    col("OrderID"),
    col("Item.Product").alias("Product"),
    col("Item.Quantity").alias("Quantity")
).show()

In [ ]:
items_df.write \
    .mode("overwrite") \
    .json("output/json/order_items")

In [ ]:
sales_df.write \
    .mode("overwrite") \
    .parquet("output/parquet/sales")

In [ ]:
parquet_df = spark.read \
    .parquet("output/parquet/sales")

parquet_df.show(truncate=False)
parquet_df.printSchema()

In [5]:
sales_df.write \
    .format("avro") \
    .mode("overwrite") \
    .save("output/avro/sales")

In [6]:
avro_df = spark.read \
    .format("avro") \
    .load("output/avro/sales")

avro_df.show()
avro_df.printSchema()

+-------+--------------------+--------+--------+-------+--------------------+-----------+
|OrderID|        CustomerName| Product|Quantity|  Price|                City|TotalAmount|
+-------+--------------------+--------+--------+-------+--------------------+-----------+
|    104|               Priya| Monitor|       2|12000.0|              Mumbai|    24000.0|
|    105|                Ravi|  Laptop|       1|60000.0|           Bangalore|    60000.0|
|    106|Rajveer Pratap Si...|  Laptop|       1|60000.0|Chikballapore, Ba...|    60000.0|
|    101|                Amit|  Laptop|       1|55000.0|              Mumbai|    55000.0|
|    102|                Neha|   Mouse|       2|  750.0|                Pune|     1500.0|
|    103|               Rahul|Keyboard|       1| 1500.0|               Delhi|     1500.0|
+-------+--------------------+--------+--------+-------+--------------------+-----------+

root
 |-- OrderID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Produ